In [1]:
import pandas as pd
import numpy as np

In [2]:
anime = pd.read_csv("anime-dataset-2023.csv")

In [3]:
anime.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24905 entries, 0 to 24904
Data columns (total 24 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   anime_id      24905 non-null  int64 
 1   Name          24905 non-null  object
 2   English name  24905 non-null  object
 3   Other name    24905 non-null  object
 4   Score         24905 non-null  object
 5   Genres        24905 non-null  object
 6   Synopsis      24905 non-null  object
 7   Type          24905 non-null  object
 8   Episodes      24905 non-null  object
 9   Aired         24905 non-null  object
 10  Premiered     24905 non-null  object
 11  Status        24905 non-null  object
 12  Producers     24905 non-null  object
 13  Licensors     24905 non-null  object
 14  Studios       24905 non-null  object
 15  Source        24905 non-null  object
 16  Duration      24905 non-null  object
 17  Rating        24905 non-null  object
 18  Rank          24905 non-null  object
 19  Popu

Model - 1 : Content based Recommendation

In [4]:
required_columns = ["anime_id", "Name", "English name", "Genres", "Synopsis", "Type", "Studios", "Source"]

df = anime[required_columns]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24905 entries, 0 to 24904
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   anime_id      24905 non-null  int64 
 1   Name          24905 non-null  object
 2   English name  24905 non-null  object
 3   Genres        24905 non-null  object
 4   Synopsis      24905 non-null  object
 5   Type          24905 non-null  object
 6   Studios       24905 non-null  object
 7   Source        24905 non-null  object
dtypes: int64(1), object(7)
memory usage: 1.5+ MB


In [5]:
df["Type"].value_counts()

Type
TV         7597
Movie      4381
OVA        4076
ONA        3533
Music      2686
Special    2558
UNKNOWN      74
Name: count, dtype: int64

In [6]:
df = df[df["Type"] != "UNKNOWN"]
df["Type"].unique()

array(['TV', 'Movie', 'OVA', 'Special', 'ONA', 'Music'], dtype=object)

In [7]:
df['Genres'] = df['Genres'].replace(",", " ")

In [8]:
df["combined_data"] = df['Genres'] + df["Type"] + df["Studios"]

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [10]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df["combined_data"])

In [11]:
tfidf_matrix.shape

(24831, 5365)

In [12]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

nn.fit(tfidf_matrix)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [13]:
anime_index = 10

distances, indices = nn.kneighbors(
    tfidf_matrix[anime_index],
    n_neighbors=10
)

print(indices)

[[   10   131 12426   245 16611 23719  1574  1969   368   141]]


In [14]:
for i in indices[0]:
    print(df['English name'].iloc[i])

Naruto
The Twelve Kingdoms
Boruto: Naruto Next Generations
Bleach
Bleach: Thousand-Year Blood War
Bleach: Thousand-Year Blood War - The Separation
Naruto Shippuden
Wonderful Adventures of Nils
Yu Yu Hakusho: Ghost Files
UNKNOWN


In [15]:
title_to_index = pd.Series(df.index, index=df["Name"].str.lower()).to_dict()
title_to_index

{'cowboy bebop': 0,
 'cowboy bebop: tengoku no tobira': 1,
 'trigun': 2,
 'witch hunter robin': 3,
 'bouken ou beet': 4,
 'eyeshield 21': 5,
 'hachimitsu to clover': 6,
 'hungry heart: wild striker': 7,
 'initial d fourth stage': 8,
 'monster': 9,
 'naruto': 10,
 'one piece': 11,
 'tennis no ouji-sama': 12,
 'ring ni kakero 1': 13,
 'school rumble': 14,
 'sunabouzu': 15,
 'texhnolyze': 16,
 'trinity blood': 17,
 'yakitate!! japan': 18,
 'zipang': 19,
 'neon genesis evangelion': 20,
 'neon genesis evangelion: death & rebirth': 21,
 'neon genesis evangelion: the end of evangelion': 22,
 'kenpuu denki berserk': 23,
 'koukaku kidoutai': 24,
 'rurouni kenshin: meiji kenkaku romantan - tsuioku-hen': 25,
 'rurouni kenshin: meiji kenkaku romantan': 26,
 'rurouni kenshin: meiji kenkaku romantan - ishinshishi e no chinkonka': 27,
 'akira': 28,
 '.hack//sign': 29,
 'aa! megami-sama!': 30,
 'aa! megami-sama! (tv)': 31,
 'tenshi kinryouku': 32,
 'kidou tenshi angelic layer': 33,
 'ai yori aoshi': 3

In [16]:
%pip install rapidfuzz

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ---------------------------------- ----- 1.3/1.5 MB 14.8 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 5.4 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [17]:
from rapidfuzz import process, fuzz

def get_best_match(title, title_to_index, score_cutoff=70):
    title = title.lower().strip()

    # Exact match
    if title in title_to_index:
        return title

    # Fuzzy match
    match = process.extractOne(
        title,
        title_to_index.keys(),
        scorer=fuzz.WRatio,
        score_cutoff=score_cutoff
    )

    if match:
        matched_title, score, _ = match
        print(f"Anime  found. Using '{matched_title}' ({score:.1f}% match)")
        return matched_title

    return None

In [46]:
def recommend(title, top_n=10):
    matched_title = get_best_match(title, title_to_index)

    if matched_title is None:
        print("Anime not found.")
        return []

    idx = title_to_index[matched_title]

    distances, indices = nn.kneighbors(
        tfidf_matrix[idx],
        n_neighbors=top_n + 1
    )

    recommendations = []

    for anime_idx in indices[0][1:]:
        recommendations.append(df.iloc[anime_idx]["Name"])

    return recommendations

   

In [40]:
print(recommend("narut",20))

Anime  found. Using 'naruto' (90.9% match)
['Juuni Kokuki', 'Boruto: Naruto Next Generations', 'Bleach', 'Bleach: Sennen Kessen-hen', 'Bleach: Sennen Kessen-hen - Ketsubetsu-tan', 'Naruto: Shippuuden', 'Nils no Fushigi na Tabi', 'Yuu☆Yuu☆Hakusho', 'Power Stone', 'Magical Hat', 'Black Clover', 'Mahou no Idol Pastel Yumi', 'Gunjou no Magmell', 'Sugar Sugar Rune', 'Dragon Quest: Yuusha Abel Densetsu', 'Tegamibachi', 'Tegamibachi Reverse', 'Shimajirou Hesoka', 'Mahou Shoujo Ore', 'Onigiri']


Model -2 : Collaberative Recommendation

In [20]:
ratings = pd.read_csv("users-score-2023.csv")

In [21]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24325191 entries, 0 to 24325190
Data columns (total 5 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   user_id      int64 
 1   Username     object
 2   anime_id     int64 
 3   Anime Title  object
 4   rating       int64 
dtypes: int64(3), object(2)
memory usage: 927.9+ MB


In [22]:
req_df = ratings[["user_id","anime_id","rating"]]
req_df.shape

(24325191, 3)

In [23]:
req_df.isnull().sum()

user_id     0
anime_id    0
rating      0
dtype: int64

In [24]:
user_counts = req_df.groupby("user_id").size()
user_counts.describe()

count    270033.000000
mean         90.082290
std         143.061117
min           1.000000
25%           8.000000
50%          36.000000
75%         122.000000
max        2986.000000
dtype: float64

In [25]:
for t in [20, 30, 50, 75, 90, 100, 122]:
    print(f"{t}: {(user_counts >= t).sum()} users")

20: 164008 users
30: 144459 users
50: 117568 users
75: 95146 users
90: 84762 users
100: 78787 users
122: 67829 users


In [26]:
for t in [150, 200, 300, 400, 500, 750, 1000]:
    print(f"{t}: {(user_counts >= t).sum()} users")

150: 56739 users
200: 41945 users
300: 10880 users
400: 6057 users
500: 4277 users
750: 1998 users
1000: 1001 users


In [27]:
active_users = user_counts[user_counts >= 300].index
filtered_df = req_df[req_df["user_id"].isin(active_users)]

filtered_df.shape

(5992274, 3)

In [28]:
anime_counts = filtered_df.groupby("anime_id").size()
anime_counts

anime_id
1        6537
5        3869
6        4312
7        1679
8         315
         ... 
55720       3
55731       1
55818     283
55821       1
55878      16
Length: 15779, dtype: int64

In [29]:
anime_counts.describe()

count    15779.000000
mean       379.762596
std        847.738229
min          1.000000
25%          6.000000
50%         49.000000
75%        302.000000
max       8992.000000
dtype: float64

In [30]:
for t in [10, 25, 50, 100, 250, 500, 1000]:
    print(f"{t}: {(anime_counts >= t).sum()} anime")

10: 11154 anime
25: 9388 anime
50: 7884 anime
100: 6253 anime
250: 4338 anime
500: 3015 anime
1000: 1788 anime


In [31]:
active_anime = anime_counts[anime_counts >= 250].index

filtered_df = filtered_df[filtered_df["anime_id"].isin(active_anime)]

filtered_df.shape

(5471334, 3)

In [32]:
filtered_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5471334 entries, 1216 to 24324240
Data columns (total 3 columns):
 #   Column    Dtype
---  ------    -----
 0   user_id   int64
 1   anime_id  int64
 2   rating    int64
dtypes: int64(3)
memory usage: 167.0 MB


In [33]:
from scipy.sparse import csr_matrix

user_codes = filtered_df["user_id"].astype("category").cat.codes
anime_codes = filtered_df["anime_id"].astype("category").cat.codes

sparse_matrix = csr_matrix(
    (
        filtered_df["rating"].astype("float32"),
        (anime_codes, user_codes)
    )
)

In [34]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=100, random_state=42)

anime_embeddings = svd.fit_transform(sparse_matrix)

In [35]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(n_neighbors=11, algorithm="brute", metric="cosine")

knn.fit(anime_embeddings)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",11
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [36]:
# Create mapping between embedding index and anime_id
anime_categories = filtered_df["anime_id"].astype("category").cat.categories

code_to_anime = dict(enumerate(anime_categories))
anime_to_code = {anime_id: code for code, anime_id in code_to_anime.items()}

# Create mapping from anime_id to title
anime_id_to_name = df.set_index("anime_id")["Name"].to_dict()

# Create mapping from title to anime_id
name_to_anime_id = df.assign(Name=df["Name"].str.lower().str.strip()).set_index("Name")["anime_id"].to_dict()

In [47]:
def recommend_anime(title, n_recommendations=10):
    title = title.lower().strip()
    title = get_best_match(title, title_to_index)

    if title is None:
        print("Anime not found.")
        return []

    anime_id = name_to_anime_id[title]

    if anime_id not in anime_to_code:
        print("Anime not present in the collaborative filtering dataset.")
        return []

    query_index = anime_to_code[anime_id]

    distances, indices = knn.kneighbors(
        anime_embeddings[query_index].reshape(1, -1),
        n_neighbors=n_recommendations + 1
    )

    recommendations = []

    for idx in indices[0][1:]:
        rec_anime_id = code_to_anime[idx]
        recommendations.append(anime_id_to_name.get(rec_anime_id, "Unknown"))

    return recommendations

In [42]:
print(recommend_anime("narut"))

Anime  found. Using 'naruto' (90.9% match)
['Naruto: Shippuuden', 'Bleach', 'Soul Eater', 'Death Note', 'Naruto Movie 1: Dai Katsugeki!! Yuki Hime Shinobu Houjou Dattebayo!', 'One Piece', 'Fullmetal Alchemist', 'Shingeki no Kyojin', 'Fairy Tail', 'Fullmetal Alchemist: Brotherhood']


Hybrid Function (Content - based + Collaberative)

In [43]:
from collections import defaultdict

def hybrid_recommend(title, top_n=10):

    content_results = recommend(title, top_n=20)
    collab_results = recommend_anime(title, n_recommendations=20)

    scores = defaultdict(float)

    # Content weight = 40%
    for rank, anime in enumerate(content_results):
        scores[anime] += (20 - rank) * 0.4

    # Collaborative weight = 60%
    for rank, anime in enumerate(collab_results):
        scores[anime] += (20 - rank) * 0.6

    final = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    print(f"\nHybrid Recommendations for '{title}':\n")

    for anime, score in final[:top_n]:
        print(anime)

In [45]:
hybrid_recommend("Naruto",20)


Hybrid Recommendations for 'Naruto':

Bleach
Naruto: Shippuuden
Soul Eater
Death Note
Naruto Movie 1: Dai Katsugeki!! Yuki Hime Shinobu Houjou Dattebayo!
One Piece
Fullmetal Alchemist
Juuni Kokuki
Shingeki no Kyojin
Boruto: Naruto Next Generations
Fairy Tail
Bleach: Sennen Kessen-hen
Fullmetal Alchemist: Brotherhood
Bleach: Sennen Kessen-hen - Ketsubetsu-tan
Shaman King
Nils no Fushigi na Tabi
Sword Art Online
Yuu☆Yuu☆Hakusho
Power Stone
Naruto: Shippuuden Movie 1
